# Sampling using binary distance superclass to create equidistante classes
This script creates classes based on distance of distribution.

As input it has the a interger $p$ that defines the number of possible classes $k$:
$$k = 2^p$$

In [1]:
import numpy as np
import networkx as nx
from pprint import pprint

In [2]:
p = 9
k = 2**p

def get_dig_k(n, l):
    return (n>>l)%2

def dist(n, p, base):
    d = 0
    for i in range(base):
        d += abs(get_dig_k(n,i)-get_dig_k(p,i))
    return d

print(f'Example: dist(15, 17) =  {dist(15, 17, p)}')


Example: dist(15, 17) =  4


In [3]:
# Calculate all distances between classes to create the graph
dist_m = np.zeros([k,k])
for i in range(k):
    for j in range(k):
        dist_m[i,j] = dist(i, j, p)

dist_m

array([[0., 1., 1., ..., 8., 8., 9.],
       [1., 0., 2., ..., 7., 9., 8.],
       [1., 2., 0., ..., 9., 7., 8.],
       ...,
       [8., 7., 9., ..., 0., 2., 1.],
       [8., 9., 7., ..., 2., 0., 1.],
       [9., 8., 8., ..., 1., 1., 0.]])

In [4]:
# Simple test of the distance, how many neighbors (distance = 1) each class has.

dist_1 = np.zeros(int(k/8))
for i in range(int(k/8)):
    dist_1[i] = sum(dist_m[i]==1)

dist_1

array([9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9.,
       9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9.,
       9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9.,
       9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9.])

## Using graph to define the classes that are equidistants:
We use graph to find a clique with same distance: 
$$
dist(c_i, c_j) = 4 \Rightarrow 8 \; \text{cliques} \\
dist(c_i, c_j) = 2 \Rightarrow 9 \; \text{cliques} \\
$$

In [5]:
# Create the dismilarity graph for a fixed distance dist_, 
dist_ = 2
graph_dissimi = np.array(dist_m == dist_, dtype=np.int32)
G=nx.Graph(graph_dissimi)
print(len(G.edges))
fct = len(G)/20
S = nx.find_cliques(G)
s = next(S)
s = G.subgraph(s).copy()
l = len(s)
sm = s
print(len(sm))
count = 0
# Scan for the maximum clique size
while True:
    count += 1
    if count % 5000000 == 0:
        print(count, len(sm))
    if len(s) >= l:
        l = len(s)
        sm = s
    try:
        s = next(S)
    except:
        break

sm = G.subgraph(sm).copy()
print(count)
print(len(sm))

# double checking the distance
dist_n = np.zeros([len(sm),len(sm)])
for i, n1 in enumerate(sm.nodes):
    for j, n2 in enumerate(sm.nodes):
        dist_n[i,j] = dist(n1, n2, p)

dist_n

9216
4
11264
9


array([[0., 2., 2., 2., 2., 2., 2., 2., 2.],
       [2., 0., 2., 2., 2., 2., 2., 2., 2.],
       [2., 2., 0., 2., 2., 2., 2., 2., 2.],
       [2., 2., 2., 0., 2., 2., 2., 2., 2.],
       [2., 2., 2., 2., 0., 2., 2., 2., 2.],
       [2., 2., 2., 2., 2., 0., 2., 2., 2.],
       [2., 2., 2., 2., 2., 2., 0., 2., 2.],
       [2., 2., 2., 2., 2., 2., 2., 0., 2.],
       [2., 2., 2., 2., 2., 2., 2., 2., 0.]])

In [6]:
# Getting the main seeds
main_Seeds = sorted(list(sm.nodes))
print(main_Seeds)


[192, 320, 384, 449, 450, 452, 456, 464, 480]


In [7]:
# For each seed geet all neighbors (distance = 1), creating the set for the supper classes
classes_supersets_dict = {}
for s in main_Seeds:
    classes_supersets_dict |= {s:s}
pprint(classes_supersets_dict)

{192: 192,
 320: 320,
 384: 384,
 449: 449,
 450: 450,
 452: 452,
 456: 456,
 464: 464,
 480: 480}


In [8]:
# Testing if all digits are used

dig_0 = set()
dig_1 = set()

for c in classes_supersets_dict:
    for i in range(p):
        if get_dig_k(c,i):
            dig_1.add(i)
        else:
            dig_0.add(i)
print(dig_1)
print(dig_0)

{0, 1, 2, 3, 4, 5, 6, 7, 8}
{0, 1, 2, 3, 4, 5, 6, 7, 8}


## Sampling

 To ensure that the distance between distributions is preserved in the Euclidean space, we generate $d_{i}^{\rho} = \mathcal{N}(\mu_i^\rho ,\,\Sigma_i)$, where $\mu_i^{\rho} \in \mathbb{R}^r$ is the location and $\Sigma_i \in \mathbb{R}^{r\times r}$ is the covariance matrix. The difference between $\mathbf{d}_i^0$ and $\mathbf{d}_i^1$ lies in their locations, since we use the same covariance matrix. For this work, we set $||\mu_i^0-\mu_i^1|| = 1$ for all $i$.


In [9]:
import pandas as pd
from scipy import stats

np.random.seed(10101)

In [10]:
version = 'V1eq'
ns      = int(1e4)  # number of samples per class
ndg     = p         # number of dimension groups
ndpg    = 13         # number of dimmensions per dimension group
nd      = ndg*ndpg  # number of dimensions
# Mean and Sigman of the distribution of the two distribution,

print(version, ns, ndg, ndpg, nd)

V1eq 10000 9 13 117


In [11]:
# Createing the mu and Sigmas for each distribution
parClass = []
noiseLevel = 0.125
for _ in range(ndg):
    c0, c1 = (np.random.rand(ndpg)-.5)*2, (np.random.rand(ndpg)-.5)*2
    centers = [c0, c1/np.linalg.norm(c1)+c0]
    rot =  np.eye(ndpg)
    scl =  np.eye(ndpg)
    for d1 in range(ndpg):
        sng   = -1.0 if np.random.rand(1) > 0.5 else 1.0
        scl[d1,d1] = ((noiseLevel)*(1+sng*np.random.rand(1)*.25))[0]
        for d2 in range(d1+1,ndpg):
            rottmp =  np.eye(ndpg)
            asixth = 1/6
            sng   = -1.0 if np.random.rand(1) > 0.5 else 1.0
            theta = sng * ((np.random.rand(1) + 1) * asixth) * np.pi
            rottmp[d1,d1] =   np.cos(theta)[0]
            rottmp[d2,d2] =   np.cos(theta)[0]
            rottmp[d1,d2] =   np.sin(theta)[0]
            rottmp[d2,d1] = - np.sin(theta)[0]
            rot = np.matmul(rot, rottmp)
    parClass.append([scl, rot, centers])
print(len(parClass), len(parClass[0]), len(parClass[0][0]), len(parClass[0][2]))

9 3 13 2


In [12]:
# drawing using the distributions
dataAll =  pd.DataFrame()
for c in classes_supersets_dict:
    dataclass = pd.DataFrame()
    dataclass['class'] = np.ones(ns, dtype=np.int8)*c
    dataclass['super_class'] = np.ones(ns, dtype=np.int8)*classes_supersets_dict[c]
    for d in range(ndg):
        cnt = get_dig_k(c,d) # make a class difference
        vrbs = stats.multivariate_normal(np.zeros(ndpg)).rvs(size=ns)
        s = parClass[d][0]
        r = parClass[d][1]
        t = parClass[d][2][cnt]
        vrbs = vrbs @ s @ r + t
        columns = [f'd{d*ndpg+i}' for i in range(ndpg)]
        tmpData = pd.DataFrame(columns=columns, data = vrbs)
        dataclass = pd.concat([dataclass, tmpData], axis=1)
    dataAll = pd.concat([dataAll, dataclass],ignore_index=True)

# saving the points
dataAll
fileName = f'../data/{version}_binaryTreeMultiDim_distSuperClasses_ns_{ns}_ndg_{ndg}_ndpg_{ndpg}.csv'
dataAll.to_csv(fileName, index=False)
print(f'File saved: {fileName}')

File saved: ../data/V1eq_binaryTreeMultiDim_distSuperClasses_ns_10000_ndg_9_ndpg_13.csv
